# 🧭 CDM Compass
**Clinical Data Management Knowledge Navigator**

Ask questions about your CDM/CDS documents in plain English and get answers with inline citations.
Documents are classified by folder. Text is chunked by document type: structured documents use
hierarchical chunking; narrative documents use paragraph chunking. Tables are converted to
readable sentences so their content is retrievable. The interactive loop remembers your last
few questions within a session.

### 📋 How to use this notebook
Run each block **from top to bottom**, in order. Re-run from Block 3 onwards after adding files.

| Block | What it does | Run again? |
|-------|-------------|------------|
| 1 | Install packages | Only once |
| 2 | Import libraries | Each session |
| 3 | Load documents | When you add files |
| 4 | Chunk text (type-aware) | When you add files |
| 5 | Build search index | When you add files |
| 6 | Connect to Ollama LLM | Each session |
| 7 | Ask a question | Anytime! |
| 8 | Interactive Q&A loop (with memory) | Anytime! |

### 📁 Document authority
```
data/raw/
├── regulatory/    <- binding requirements (ICH, FDA, EMA)
└── opinion/       <- position papers, white papers (SCDM, JSCDM)
```


## 📦 Block 1 -- Install Required Packages
Run this once.

> ⚠️ **Before running Block 6**, install Ollama:
> 1. Go to [https://ollama.com](https://ollama.com) and download the app
> 2. Open Terminal and run: `ollama pull qwen2.5:14b`


In [1]:
import sys

!{sys.executable} -m pip install --upgrade pip --quiet
!{sys.executable} -m pip install "sentence-transformers[cross-encoder]" chromadb pypdf pdfplumber openpyxl python-docx python-pptx rank-bm25 requests tqdm --quiet
!{sys.executable} -m pip install --upgrade jupyter ipywidgets --quiet

print("\u2705 All packages installed!")


✅ All packages installed!


## 📚 Block 2 -- Import Libraries
All imports live here. Run at the start of every session.


In [2]:
import os
import re
import json
import csv
import textwrap
import xml.etree.ElementTree as ET
import requests
import numpy as np
import torch
import chromadb
import pdfplumber
from rank_bm25 import BM25Okapi

from pypdf import PdfReader
from pptx import Presentation
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.notebook import tqdm
import openpyxl
import docx  # python-docx

print("\u2705 All libraries imported successfully!")


✅ All libraries imported successfully!


## 📁 Block 3 -- Load Documents

**👇 Only change `DOCS_FOLDER`** if your documents are in a different location.

| Subfolder | Authority | Use for |
|-----------|-----------|---------|
| `regulatory/` | ⚖️ Regulatory | ICH, FDA, EMA -- binding requirements |
| `opinion/` | 💬 Opinion | SCDM, JSCDM, white papers, position papers |

**PDF table handling:** tables in PDFs are extracted with `pdfplumber` and converted to
readable sentences (e.g. `'Parameter: dose, Unit: mg, Value: 10.'`) so their content
is searchable. Plain prose pages use standard text extraction.

**Supported:** PDF, PPTX, XLSX, DOCX, JSON, XML, CSV, TXT

> ⚠️ Old-format `.ppt` files are not supported. Open in PowerPoint → Save As → .pptx first.


In [3]:
# ============================================================
# 🔴 CHANGE THIS to your documents folder path if needed
# ============================================================
DOCS_FOLDER = "../data/raw/"
# ============================================================

AUTHORITY_FOLDERS = {'regulatory': 'regulatory', 'opinion': 'opinion'}


def get_authority(filepath):
    """Derive authority level from the subfolder the file lives in."""
    parts = filepath.replace('\\', '/').split('/')
    for part in parts:
        if part.lower() in AUTHORITY_FOLDERS:
            return AUTHORITY_FOLDERS[part.lower()]
    raise ValueError(
        f"\n\u274c Cannot classify: '{os.path.basename(filepath)}'\n"
        f"   Move it into one of these subfolders:\n"
        f"     {DOCS_FOLDER}regulatory/   <- binding requirements (ICH, FDA, EMA)\n"
        f"     {DOCS_FOLDER}opinion/       <- position papers and white papers"
    )


def extract_heading(text):
    """Try to find a heading on the page (short ALL-CAPS line)."""
    for line in text.split('\n'):
        stripped = line.strip()
        if 4 < len(stripped) < 80 and stripped.isupper():
            return stripped
    return 'Unknown'


def extract_pdf_tables(filepath):
    """
    Extract tables from a PDF using pdfplumber and convert each row to a
    readable sentence: 'Header1: value1, Header2: value2.'
    Returns a list of sentence strings, one per table row.
    Tables in regulatory documents (ICH, FDA) often contain the most precise
    data -- this ensures that data is retrievable rather than silently lost.
    """
    sentences = []
    try:
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                for table in page.extract_tables():
                    if not table or len(table) < 2:
                        continue
                    headers = [str(h).strip() if h else '' for h in table[0]]
                    for row in table[1:]:
                        pairs = [
                            f'{h}: {str(v).strip()}'
                            for h, v in zip(headers, row)
                            if h and v and str(v).strip()
                        ]
                        if pairs:
                            sentences.append(', '.join(pairs) + '.')
    except Exception:
        pass  # table extraction is best-effort; prose extraction continues
    return sentences


def load_pdf(filepath):
    """Load a PDF. Returns one entry per page plus table rows as separate entries."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        # Standard prose extraction (page by page)
        reader = PdfReader(filepath)
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                docs.append({
                    'text':      text.strip(),
                    'source':    filename,
                    'page':      i + 1,
                    'heading':   extract_heading(text),
                    'authority': authority,
                    'doc_type':  'prose',
                })
        # Table extraction: each row becomes its own retrievable entry
        table_sentences = extract_pdf_tables(filepath)
        if table_sentences:
            docs.append({
                'text':      '\n'.join(table_sentences),
                'source':    filename,
                'page':      'tables',
                'heading':   'Extracted Tables',
                'authority': authority,
                'doc_type':  'table',
            })
    except Exception as e:
        print(f'  \u274c Error reading PDF {filename}: {e}')
    return docs


def detect_doc_type(text):
    """
    Detect whether a document is structured (has clear heading hierarchy)
    or narrative (flowing prose).

    Heuristic: count lines that look like headings (short, possibly numbered,
    ending without a period). If heading density is above threshold, the doc
    has meaningful structure worth preserving hierarchically.
    """
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if not lines:
        return 'narrative'
    heading_count = sum(
        1 for l in lines
        if len(l) < 80
        and not l.endswith('.')
        and (l.isupper() or re.match(r'^(\d+\.)+\s+\w', l) or re.match(r'^[A-Z][^.]{3,60}$', l))
    )
    # More than 8% of lines look like headings -> structured document
    return 'structured' if heading_count / len(lines) > 0.08 else 'narrative'


def load_pptx(filepath):
    """Load a PowerPoint file. Extracts text from slides and speaker notes."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        prs = Presentation(filepath)
        for i, slide in enumerate(prs.slides, 1):
            parts = [shape.text_frame.text.strip()
                     for shape in slide.shapes if shape.has_text_frame]
            if slide.has_notes_slide:
                notes = slide.notes_slide.notes_text_frame.text.strip()
                if notes:
                    parts.append(f'[Speaker notes] {notes}')
            text = '\n'.join(p for p in parts if p)
            if text:
                docs.append({'text': text, 'source': filename,
                             'page': i, 'heading': f'Slide {i}',
                             'authority': authority, 'doc_type': 'narrative'})
    except Exception as e:
        print(f'  \u274c Error reading PPTX {filename}: {e}')
    return docs


def load_xlsx(filepath):
    """Load an Excel file. Converts each row to a readable sentence."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
        for sheet_name in wb.sheetnames:
            ws        = wb[sheet_name]
            all_rows  = list(ws.iter_rows(values_only=True))
            if not all_rows:
                continue
            # First non-empty row is treated as headers
            headers = [str(c).strip() if c is not None else '' for c in all_rows[0]]
            sentences = []
            for row in all_rows[1:]:
                pairs = [
                    f'{h}: {str(v).strip()}'
                    for h, v in zip(headers, row)
                    if h and v is not None and str(v).strip()
                ]
                if pairs:
                    sentences.append(', '.join(pairs) + '.')
            if sentences:
                docs.append({'text': '\n'.join(sentences), 'source': filename,
                             'page': sheet_name, 'heading': f'Sheet: {sheet_name}',
                             'authority': authority, 'doc_type': 'table'})
    except Exception as e:
        print(f'  \u274c Error reading XLSX {filename}: {e}')
    return docs


def load_docx(filepath):
    """Load a Word document."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        document   = docx.Document(filepath)
        paragraphs = [p.text.strip() for p in document.paragraphs if p.text.strip()]
        full_text  = '\n'.join(paragraphs)
        if full_text:
            docs.append({'text': full_text, 'source': filename, 'page': 1,
                         'heading': paragraphs[0][:80] if paragraphs else 'Unknown',
                         'authority': authority,
                         'doc_type': detect_doc_type(full_text)})
    except Exception as e:
        print(f'  \u274c Error reading DOCX {filename}: {e}')
    return docs


def load_json(filepath):
    """Load a JSON file."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        docs.append({'text': json.dumps(data, indent=2, ensure_ascii=False),
                     'source': filename, 'page': 1, 'heading': 'JSON Document',
                     'authority': authority, 'doc_type': 'narrative'})
    except Exception as e:
        print(f'  \u274c Error reading JSON {filename}: {e}')
    return docs


def load_xml(filepath):
    """Load an XML file."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        root  = ET.parse(filepath).getroot()
        texts = [f'{e.tag}: {e.text.strip()}' for e in root.iter()
                 if e.text and e.text.strip()]
        full_text = '\n'.join(texts)
        if full_text:
            docs.append({'text': full_text, 'source': filename, 'page': 1,
                         'heading': f'XML Root: {root.tag}',
                         'authority': authority, 'doc_type': 'narrative'})
    except Exception as e:
        print(f'  \u274c Error reading XML {filename}: {e}')
    return docs


def load_csv(filepath):
    """Load a CSV file. Converts each row to a readable sentence."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            reader  = csv.reader(f)
            all_rows = list(reader)
        if not all_rows:
            return docs
        headers = [str(c).strip() for c in all_rows[0]]
        sentences = []
        for row in all_rows[1:]:
            pairs = [f'{h}: {v.strip()}' for h, v in zip(headers, row)
                     if h and v.strip()]
            if pairs:
                sentences.append(', '.join(pairs) + '.')
        full_text = '\n'.join(sentences)
        if full_text:
            docs.append({'text': full_text, 'source': filename, 'page': 1,
                         'heading': 'CSV Document',
                         'authority': authority, 'doc_type': 'table'})
    except Exception as e:
        print(f'  \u274c Error reading CSV {filename}: {e}')
    return docs


def load_txt(filepath):
    """Load a plain text file."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            text = f.read().strip()
        if text:
            docs.append({'text': text, 'source': filename, 'page': 1,
                         'heading': 'Text Document',
                         'authority': authority,
                         'doc_type': detect_doc_type(text)})
    except Exception as e:
        print(f'  \u274c Error reading TXT {filename}: {e}')
    return docs


LOADERS = {
    '.pdf':  load_pdf,  '.pptx': load_pptx,
    '.xlsx': load_xlsx, '.xls':  load_xlsx,
    '.docx': load_docx, '.json': load_json,
    '.xml':  load_xml,  '.csv':  load_csv,
    '.txt':  load_txt,
}


def load_all_documents(folder_path):
    """Walk the folder tree and load all supported files."""
    all_docs, skipped = [], []

    if not os.path.exists(folder_path):
        print(f'\u274c Folder not found: {folder_path}')
        print('   Update DOCS_FOLDER above.')
        return all_docs

    for root, dirs, files in os.walk(folder_path):
        for filename in sorted(files):
            ext       = os.path.splitext(filename)[1].lower()
            full_path = os.path.join(root, filename)

            if ext == '.ppt':
                print(f'  \u26a0\ufe0f  Skipped (old format): {filename}')
                print(f'     -> Save as .pptx in PowerPoint first.')
                skipped.append(filename)
                continue

            if ext in LOADERS:
                try:
                    print(f'  \U0001f4c4 Loading: {filename}')
                    docs = LOADERS[ext](full_path)
                    auth = get_authority(full_path).upper()
                    # Show detected doc_type for PDFs and DOCX
                    types = set(d.get('doc_type','?') for d in docs)
                    print(f'     -> {len(docs)} section(s)  [{auth}]  types: {", ".join(sorted(types))}')
                    all_docs.extend(docs)
                except ValueError as e:
                    print(e)
                    print('\n\u26d4 Loading stopped. Classify the file and re-run Block 3.')
                    return all_docs
            elif not filename.startswith('.'):
                skipped.append(filename)

    print(f'\n\u2705 Total sections loaded: {len(all_docs)}')
    if skipped:
        print(f'\u26a0\ufe0f  Skipped: {", ".join(skipped)}')
    return all_docs


print(f'\U0001f50d Scanning folder: {DOCS_FOLDER}\n')
docs = load_all_documents(DOCS_FOLDER)


🔍 Scanning folder: ../data/raw/

  📄 Loading: LOINC Public Review Comments.xlsx
     -> 2 section(s)  [OPINION]  types: table
  📄 Loading: LOINC to LB Mapping-Read Me.pdf
     -> 5 section(s)  [OPINION]  types: prose, table
  📄 Loading: LOINC_to_LB_Mapping Document_FINAL.csv
     -> 1 section(s)  [OPINION]  types: table
  📄 Loading: LOINC_to_LB_Mapping Document_FINAL.xlsx
     -> 2 section(s)  [OPINION]  types: table
  📄 Loading: Guidance for eCOA Development in Clinical Trials.pdf
     -> 23 section(s)  [OPINION]  types: prose
  📄 Loading: jscdm-116-lebedys.pdf
     -> 20 section(s)  [OPINION]  types: prose
  📄 Loading: jscdm-117-hills.pdf
     -> 12 section(s)  [OPINION]  types: prose
  📄 Loading: jscdm-118-amatya.pdf
     -> 13 section(s)  [OPINION]  types: prose
  📄 Loading: jscdm-20-stokman.pdf
     -> 8 section(s)  [OPINION]  types: prose
  📄 Loading: jscdm-260-king.pdf
     -> 10 section(s)  [OPINION]  types: prose, table
  📄 Loading: jscdm-262-king.pdf
     -> 11 section(s)  [O

Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)


     -> 15 section(s)  [OPINION]  types: prose, table
  📄 Loading: The Uncomfortable Conversation AI And Data Use In Clinical Trials.pdf
     -> 3 section(s)  [OPINION]  types: prose
  📄 Loading: 45966642fnl.pdf
     -> 16 section(s)  [REGULATORY]  types: prose, table
  📄 Loading: 58687461fnlrv8_1.pdf
     -> 24 section(s)  [REGULATORY]  types: prose, table
  📄 Loading: ectd_tech_guide_v1.9.pdf
     -> 32 section(s)  [REGULATORY]  types: prose, table
  📄 Loading: fda-dscv11.0.xlsx
     -> 6 section(s)  [REGULATORY]  types: table
  📄 Loading: sdTCG-v6.1-December-2025-FINAL.pdf
     -> 105 section(s)  [REGULATORY]  types: prose, table
  📄 Loading: E11_R1_Addendum.pdf
     -> 26 section(s)  [REGULATORY]  types: prose, table
  📄 Loading: E2A_Guideline.pdf
     -> 13 section(s)  [REGULATORY]  types: prose, table
  📄 Loading: E9-R1_Step4_Guideline_2019_1203.pdf
     -> 23 section(s)  [REGULATORY]  types: prose, table
  📄 Loading: E9_Guideline.pdf
     -> 40 section(s)  [REGULATORY]  types: p

## ✂️ Block 4 -- Type-Aware Chunking

Each document is chunked according to its detected type:

| Doc type | Strategy | Best for |
|----------|----------|---------|
| `structured` | **Hierarchical** -- section context preserved | ICH guidelines, numbered specs |
| `narrative` | **Paragraph** -- complete semantic units | Position papers, prose docs |
| `table` | **Row sentences** -- already converted in Block 3 | XLSX, CSV, PDF tables |

**Hierarchical chunking** stores each paragraph as a leaf chunk but also keeps its parent
section text. At retrieval time, if multiple paragraphs from the same section are relevant,
the section is promoted -- ensuring the LLM sees complete context rather than isolated fragments.

| Setting | Default | Effect |
|---------|---------|--------|
| `MIN_PARA_WORDS` | 30 | Merge paragraphs shorter than this with the next |
| `MAX_PARA_WORDS` | 400 | Split paragraphs longer than this at sentence boundaries |
| `HIER_SECTION_WORDS` | 800 | Max words in a hierarchical section chunk |


In [4]:
# ============================================================
# Settings
# ============================================================
MIN_PARA_WORDS    = 30    # merge paragraphs shorter than this
MAX_PARA_WORDS    = 400   # split paragraphs longer than this
HIER_SECTION_WORDS = 800  # max words in a hierarchical section chunk
# ============================================================

chunks   = []  # text of each chunk
metadata = []  # source, page, heading, authority, chunk_type per chunk


# ── Shared helpers ────────────────────────────────────────────────────────

def split_sentences(text):
    """Split text into sentences."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


# ── Strategy 1: Paragraph chunking (narrative docs) ───────────────────────

def paragraph_chunks(text):
    """
    Split on blank lines, merge short paragraphs forward, split long ones
    at sentence boundaries, add one-sentence overlap at each boundary.
    """
    raw = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

    # Merge short paragraphs forward
    merged, buf = [], ''
    for para in raw:
        if buf:
            combined = buf + ' ' + para
            if len(buf.split()) < MIN_PARA_WORDS:
                buf = combined
            else:
                merged.append(buf)
                buf = para
        else:
            buf = para
    if buf:
        merged.append(buf)

    # Split long paragraphs at sentence boundaries
    sized = []
    for para in merged:
        if len(para.split()) <= MAX_PARA_WORDS:
            sized.append(para)
        else:
            sents = split_sentences(para)
            cur_words, cur_sents = 0, []
            for sent in sents:
                w = len(sent.split())
                if cur_words + w > MAX_PARA_WORDS and cur_sents:
                    sized.append(' '.join(cur_sents))
                    cur_sents, cur_words = [sent], w
                else:
                    cur_sents.append(sent)
                    cur_words += w
            if cur_sents:
                sized.append(' '.join(cur_sents))

    # Add one-sentence overlap from next chunk
    result = []
    for idx, chunk in enumerate(sized):
        if idx + 1 < len(sized):
            next_sents = split_sentences(sized[idx + 1])
            if next_sents:
                chunk = chunk + ' ' + next_sents[0]
        result.append(chunk)
    return result


# ── Strategy 2: Hierarchical chunking (structured docs) ──────────────────

def hierarchical_chunks(text):
    """
    Split structured documents (numbered sections, clear headings) into
    two levels:
      - Leaf chunks: individual paragraphs (for precise retrieval)
      - Section chunks: groups of paragraphs under the same heading
        (for broad context when multiple leaves match)

    Both levels are added to the index. Section chunks are tagged
    'hier_section'; leaf chunks are tagged 'hier_leaf'.
    Returns a list of (text, chunk_type) tuples.
    """
    # Detect section boundaries: lines that look like headings
    heading_pat = re.compile(
        r'^(\d+\.)+\s+\w|^[A-Z][A-Z0-9 ]{3,60}$|^[A-Z][^.]{3,60}$'
    )
    lines   = text.split('\n')
    sections = []  # list of (heading_line, [body_lines])
    current_heading = 'Introduction'
    current_body    = []

    for line in lines:
        stripped = line.strip()
        if stripped and len(stripped) < 80 and heading_pat.match(stripped):
            if current_body:
                sections.append((current_heading, current_body))
            current_heading = stripped
            current_body    = []
        elif stripped:
            current_body.append(stripped)
    if current_body:
        sections.append((current_heading, current_body))

    result = []
    for heading, body_lines in sections:
        body = '\n'.join(body_lines)
        if not body.strip():
            continue

        # Section-level chunk (broad context)
        section_words = body.split()
        if len(section_words) <= HIER_SECTION_WORDS:
            result.append((f'{heading}\n{body}', 'hier_section'))
        else:
            # Section too long: truncate to HIER_SECTION_WORDS
            result.append((' '.join(section_words[:HIER_SECTION_WORDS]), 'hier_section'))

        # Leaf chunks: individual paragraphs within the section
        for leaf in paragraph_chunks(body):
            result.append((f'{heading}\n{leaf}', 'hier_leaf'))

    return result


# ── Strategy 3: Table rows (already sentence-ified in Block 3) ───────────

def table_chunks(text):
    """Each line is already a readable sentence from Block 3 -- keep as-is."""
    return [(line.strip(), 'table') for line in text.split('\n') if line.strip()]


# ── Router ───────────────────────────────────────────────────────────────

def chunk_document(doc):
    """Route each document to the right chunking strategy."""
    doc_type = doc.get('doc_type', 'narrative')
    text     = doc['text']

    if doc_type == 'table':
        pairs = table_chunks(text)
    elif doc_type == 'structured':
        pairs = hierarchical_chunks(text)
    else:
        pairs = [(c, 'paragraph') for c in paragraph_chunks(text)]

    result = []
    for chunk_text, chunk_type in pairs:
        if chunk_text.strip():
            result.append((chunk_text, chunk_type))
    return result


# ── Build chunk + metadata lists ─────────────────────────────────────────

type_counts = {}
for doc in docs:
    for chunk_text, chunk_type in chunk_document(doc):
        chunks.append(chunk_text)
        metadata.append({
            'source':     doc['source'],
            'page':       doc['page'],
            'heading':    doc['heading'],
            'authority':  doc['authority'],
            'chunk_type': chunk_type,
        })
        type_counts[chunk_type] = type_counts.get(chunk_type, 0) + 1

word_counts = [len(c.split()) for c in chunks]
print(f'\u2705 Created {len(chunks)} chunks from {len(docs)} document sections')
print(f'   Chunk type breakdown:')
for t, n in sorted(type_counts.items()):
    print(f'     {t:<20} {n}')
print(f'   Word count -- min: {min(word_counts)}  median: {int(np.median(word_counts))}  max: {max(word_counts)}')


✅ Created 48248 chunks from 3658 document sections
   Chunk type breakdown:
     hier_leaf            11
     hier_section         9
     paragraph            9100
     table                39128
   Word count -- min: 1  median: 79  max: 800


## 🧠 Block 5 -- Build the Search Index

| Component | Type | Role |
|-----------|------|------|
| **ChromaDB** | Semantic (vector) | Finds chunks with similar *meaning* |
| **BM25** | Keyword | Finds chunks with exact *words* |
| **CrossEncoder** | Reranker | Re-scores top candidates with higher precision |

**This may take a few minutes** -- models are cached after the first run.


In [5]:
# ── Step 1: Load the embedding model ─────────────────────────────────────────
print("⏳ Loading embedding model (downloads once, then cached)...")
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"   Using device: {device}")

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("✅ Embedding model ready!")

# ── Step 2: Encode all chunks into embeddings ─────────────────────────────────
print(f"\n⏳ Encoding {len(chunks)} chunks into embeddings...")
embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True,
)
embeddings = embeddings.astype("float32")
print(f"✅ Embeddings created: {embeddings.shape} (chunks × dimensions)")

# ── Step 3: Build ChromaDB semantic index ─────────────────────────────────────
print("\n⏳ Building semantic index (ChromaDB)...")
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("rag_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="rag_docs",
    metadata={"hnsw:space": "cosine"},
)

# Add in batches — ChromaDB has a max batch size of 5461
BATCH_SIZE = 5000
total      = len(chunks)

for start in range(0, total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total)
    collection.add(
        ids        = [str(i) for i in range(start, end)],
        embeddings = embeddings[start:end].tolist(),
        documents  = chunks[start:end],
        metadatas  = metadata[start:end],
    )
    print(f"   Indexed {end}/{total} chunks...")

print(f"✅ Semantic index ready with {collection.count()} vectors")

# ── Step 4: Build BM25 keyword index ─────────────────────────────────────────
print("\n⏳ Building keyword index (BM25)...")
tokenised_chunks = [chunk.lower().split() for chunk in chunks]
bm25_index       = BM25Okapi(tokenised_chunks)
print(f"✅ Keyword index ready over {len(chunks)} chunks")
print("\n🎉 Both indexes built — hybrid search is ready!")

# -- Step 5: Load the cross-encoder reranker --
print("\n\u23f3 Loading reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2", max_length=512)
print("\u2705 Reranker ready!")
print("\n\U0001f389 CDM Compass is ready!")


⏳ Loading embedding model (downloads once, then cached)...
   Using device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready!

⏳ Encoding 48248 chunks into embeddings...


Batches:   0%|          | 0/1508 [00:00<?, ?it/s]

✅ Embeddings created: (48248, 384) (chunks × dimensions)

⏳ Building semantic index (ChromaDB)...
   Indexed 5000/48248 chunks...
   Indexed 10000/48248 chunks...
   Indexed 15000/48248 chunks...
   Indexed 20000/48248 chunks...
   Indexed 25000/48248 chunks...
   Indexed 30000/48248 chunks...
   Indexed 35000/48248 chunks...
   Indexed 40000/48248 chunks...
   Indexed 45000/48248 chunks...
   Indexed 48248/48248 chunks...
✅ Semantic index ready with 48248 vectors

⏳ Building keyword index (BM25)...
✅ Keyword index ready over 48248 chunks

🎉 Both indexes built — hybrid search is ready!

⏳ Loading reranker...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Reranker ready!

🎉 CDM Compass is ready!


## 🤖 Block 6 -- Connect to Ollama (Local AI)

| Model | Pull command | Notes |
|-------|-------------|-------|
| `qwen2.5:14b` | `ollama pull qwen2.5:14b` | Best citation reliability ✅ |
| `mistral` | `ollama pull mistral` | Good alternative |
| `llama3.2` | `ollama pull llama3.2` | Also solid |
| `phi3` | `ollama pull phi3` | Fastest -- least reliable on citations |


In [6]:
OLLAMA_MODEL = "qwen2.5:14b"
# Alternatives: mistral  |  llama3.2  |  phi3
OLLAMA_URL   = "http://localhost:11434/api/generate"


def ask_ollama(prompt, model=OLLAMA_MODEL, temperature=0.3):
    """Send a prompt to Ollama and return the response text."""
    try:
        response = requests.post(
            OLLAMA_URL,
            json={'model': model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': temperature}},
            timeout=120,
        )
        response.raise_for_status()
        return response.json().get('response', '').strip()
    except requests.exceptions.ConnectionError:
        return (
            '\u274c Cannot connect to Ollama.\n'
            '   Download from: https://ollama.com\n'
            f'   Then run: ollama pull {OLLAMA_MODEL}'
        )
    except Exception as e:
        return f'\u274c Error: {e}'


print(f"\U0001f517 Testing Ollama (model: {OLLAMA_MODEL})...")
test_reply = ask_ollama("Reply with exactly: 'Ollama is working!'")
print(f"Response: {test_reply}")
if "working" in test_reply.lower() or "ollama" in test_reply.lower():
    print("\n\u2705 Ollama ready!")
else:
    print("\n\u26a0\ufe0f  Got a response -- Ollama seems to be running.")


🔗 Testing Ollama (model: qwen2.5:14b)...
Response: Ollama is working!

✅ Ollama ready!


## 🔍 Block 7 -- Search + Ask a Question

Four-stage pipeline per question:

1. **Hybrid search** -- ChromaDB (semantic) + BM25 (keyword), fused by weight
2. **Reranking** -- cross-encoder selects the best `FINAL_K` from `TOP_K` candidates
3. **Compression** -- only the most relevant sentences per chunk reach the LLM
4. **Context-aware answer** -- last `HISTORY_TURNS` Q&A pairs included for follow-ups

Sources appear in the order cited. Gaps (e.g. [1],[2],[4]) = retrieved but not used.

**👇 Change `MY_QUESTION` to whatever you want to ask.**


In [7]:
# ============================================================
# 👇 Settings
# ============================================================
MY_QUESTION      = "What skills does an experienced Clinical Data Manager of 25+ years need to transition to Data Science?"
TOP_K            = 10
FINAL_K          = 5
HISTORY_TURNS    = 3
TOKEN_WARN_LIMIT = 3000
SEMANTIC_WEIGHT  = 0.7
KEYWORD_WEIGHT   = 0.3
# ============================================================


def _c(code, text): return f"\033[{code}m{text}\033[0m"
def green(t):   return _c('32;1', t)
def yellow(t):  return _c('33;1', t)
def red(t):     return _c('31;1', t)
def magenta(t): return _c('35;1', t)
def blue(t):    return _c('34;1', t)
def bold(t):    return _c('1',    t)
def dim(t):     return _c('2',    t)


def estimate_tokens(text): return len(text) // 4


def hybrid_search(question, top_k=10):
    """Retrieve candidates using semantic + keyword search fused by weight."""
    query_embedding  = embed_model.encode([question]).astype('float32')
    sem_results      = collection.query(
        query_embeddings=query_embedding.tolist(), n_results=top_k)
    sem_scores = {
        int(idx): 1.0 - dist
        for idx, dist in zip(sem_results['ids'][0], sem_results['distances'][0])
    }
    tokenised_query  = question.lower().split()
    bm25_raw         = bm25_index.get_scores(tokenised_query)
    ceiling          = np.percentile(bm25_raw[bm25_raw > 0], 95) if (bm25_raw > 0).any() else 1.0
    bm25_norm        = np.clip(bm25_raw / ceiling, 0.0, 1.0)
    bm25_top_indices = np.argsort(bm25_norm)[::-1][:top_k]
    bm25_scores      = {int(i): float(bm25_norm[i]) for i in bm25_top_indices}
    fused = []
    for i in set(sem_scores.keys()) | set(bm25_scores.keys()):
        s = sem_scores.get(i, 0.0)
        k = bm25_scores.get(i, 0.0)
        fused.append({
            'text':         chunks[i],
            'source':       metadata[i]['source'],
            'page':         metadata[i]['page'],
            'heading':      metadata[i]['heading'],
            'authority':    metadata[i]['authority'],
            'chunk_type':   metadata[i].get('chunk_type', 'paragraph'),
            'hybrid_score': round(1.0 - (SEMANTIC_WEIGHT * s + KEYWORD_WEIGHT * k), 3),
            'sem_score':    round(s, 3),
            'kw_score':     round(k, 3),
        })
    fused.sort(key=lambda x: (
        0 if x['authority'] == 'regulatory' else 1, x['hybrid_score']))
    return fused[:top_k]


def rerank(question, candidates, final_k=5):
    """Re-score with cross-encoder; regulatory sources surface first."""
    scores = reranker.predict([(question, c['text']) for c in candidates])
    for c, score in zip(candidates, scores):
        c['rerank_score'] = float(score)
    candidates.sort(key=lambda x: (
        0 if x['authority'] == 'regulatory' else 1, -x['rerank_score']))
    return candidates[:final_k]


def compress(question, chunk_text, min_sentences=2):
    """Return only the sentences most relevant to the question."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', chunk_text) if s.strip()]
    if len(sentences) <= min_sentences:
        return chunk_text
    scores    = reranker.predict([(question, s) for s in sentences])
    threshold = float(np.mean(scores))
    kept      = [s for s, sc in zip(sentences, scores) if sc >= threshold]
    if len(kept) < min_sentences:
        top_idx = np.argsort(scores)[::-1][:min_sentences]
        kept    = [sentences[i] for i in sorted(top_idx)]
    return ' '.join(kept)


def build_prompt(question, retrieved_chunks, history=None):
    """Build the LLM prompt with compressed chunks and optional history."""
    seen, unique = set(), []
    for chunk in retrieved_chunks:
        key = (chunk['source'], chunk['page'])
        if key not in seen:
            seen.add(key)
            unique.append(chunk)
    context_parts = []
    for i, chunk in enumerate(unique, 1):
        fname           = os.path.basename(chunk['source'])
        compressed_text = compress(question, chunk['text'])
        context_parts.append(f"[{i}] {fname} -- page {chunk['page']}\n{compressed_text}")
    context = '\n\n'.join(context_parts)
    history_block = ''
    if history:
        turns = []
        for q, a in history[-HISTORY_TURNS:]:
            a_short = a[:400] + '...' if len(a) > 400 else a
            turns.append(f'Q: {q}\nA: {a_short}')
        history_block = '=== RECENT CONVERSATION ===\n' + '\n\n'.join(turns) + '\n\n'
    return (
        'You are an expert navigator of Clinical Data Management (CDM) and Clinical Data Science (CDS).\n\n'
        'Answer using ONLY the document excerpts provided. If conversation history is present, use it for context only -- do not cite it.\n\n'
        'Language rules:\n'
        '- Regulatory (ICH, FDA, EMA): must, is required, mandates, shall.\n'
        '- Opinion (SCDM, JSCDM, position papers): recommends, suggests, proposes.\n'
        'Flag conflicts between regulatory and opinion sources explicitly.\n\n'
        'Citation rules -- follow exactly:\n'
        '- Use [1], [2], [3] only -- a number in square brackets, nothing else.\n'
        '- Never write [1: filename] or [2, page X] or any other variant.\n'
        '- Place bracket after the claim, before punctuation.\n'
        '- Do NOT add a references section. Do NOT use ibid.\n'
        '- If context is insufficient, say so clearly.\n\n'
        + history_block
        + '=== DOCUMENT CONTEXT ===\n'
        + context
        + '\n\n=== QUESTION ===\n'
        + question
        + '\n\n=== ANSWER ==='
    )


def ask_question(question, top_k=10, final_k=5, history=None):
    """CDM Compass pipeline: hybrid -> rerank -> compress -> answer -> cited sources."""
    print()
    print(bold(f'\U0001f9ed CDM Compass  --  "{question}"'))
    suffix = f' | {len(history)} turn(s) in memory' if history else ''
    print(dim(f'   {int(SEMANTIC_WEIGHT*100)}% semantic + {int(KEYWORD_WEIGHT*100)}% keyword  |  top {final_k} of {top_k}{suffix}'))

    candidates = hybrid_search(question, top_k=top_k)
    retrieved  = rerank(question, candidates, final_k=final_k)
    print(dim(f'   {len(retrieved)} chunks after reranking  (types: {", ".join(sorted(set(r["chunk_type"] for r in retrieved)))})' ))

    prompt     = build_prompt(question, retrieved, history=history)
    est        = estimate_tokens(prompt)
    if est > TOKEN_WARN_LIMIT:
        print(dim(f'   \u26a0\ufe0f  ~{est} tokens -- consider reducing HISTORY_TURNS or FINAL_K'))
    else:
        print(dim(f'   ~{est} tokens'))

    print(dim(f'   Asking {OLLAMA_MODEL}...'))
    answer = ask_ollama(prompt)

    answer = re.split(
        r'\n\s*(\[?[Rr]eferences?\]?[\s:]*$'
        r'|\[\d+\][:\s]+\S+\.(?:pdf|docx|xlsx|pptx|txt)'
        r'|\[\d+\]\s*\[)',
        answer, flags=re.MULTILINE)[0].strip()
    answer = re.sub(r'\[(\d+)[^\]]+\]', r'[\1]', answer)

    print('\n' + bold('=' * 60))
    print(bold('\U0001f4ac  ANSWER'))
    print(bold('=' * 60))
    for line in answer.split('\n'):
        if line.strip(): print(textwrap.fill(line, width=80))
        else: print()

    seen, deduped = set(), []
    for r in retrieved:
        key = (r['source'], r['page'])
        if key not in seen:
            seen.add(key)
            deduped.append(r)

    cited, cited_keys = [], set()
    for i, r in enumerate(deduped):
        fname   = os.path.basename(r['source'])
        key     = (fname, r['page'])
        pattern = re.compile(rf'\[{i + 1}[,\]\s]')
        if pattern.search(answer) and key not in cited_keys:
            cited_keys.add(key)
            cited.append((i + 1, r))

    if cited:
        print('\n' + bold('-' * 60))
        print(bold('\U0001f4da  SOURCES CITED  ') + dim('(in order cited -- gaps = retrieved but not used)'))
        print(bold('-' * 60))
        for orig_n, r in cited:
            d         = r['hybrid_score']
            authority = r.get('authority', 'opinion')
            ctype     = r.get('chunk_type', '')
            auth_badge = magenta('\u2696\ufe0f  REGULATORY') if authority == 'regulatory' \
                         else blue('\U0001f4ac  OPINION   ')
            rel_badge  = (green('\U0001f7e2 HIGH  ') if d < 0.5 else
                          yellow('\U0001f7e1 MEDIUM') if d < 0.65 else
                          red('\U0001f534 LOW   '))
            ctype_dim  = dim(f'[{ctype}]') if ctype else ''
            fname      = os.path.basename(r['source'])
            print(f'  {auth_badge}  {rel_badge}  {bold(f"[{orig_n}]")}  {ctype_dim}')
            print(f'         {bold(fname)}  --  page {r["page"]}')
            print(dim(f'         hybrid: {d:.3f}  |  rerank: {r["rerank_score"]:.2f}  |  semantic: {r["sem_score"]:.3f}  |  keyword: {r["kw_score"]:.3f}'))
            print()
    else:
        print('\n' + dim('\u26a0\ufe0f  No explicit citations found.'))
        print(dim('   Try qwen2.5:14b in Block 6, or increase TOP_K above.'))

    return answer


answer = ask_question(MY_QUESTION, top_k=TOP_K, final_k=FINAL_K)



🧭 CDM Compass  --  "What skills does an experienced Clinical Data Manager of 25+ years need to transition to Data Science?"
   70% semantic + 30% keyword  |  top 5 of 10
   5 chunks after reranking  (types: paragraph)
   ~951 tokens
   Asking qwen2.5:14b...

💬  ANSWER
An experienced Clinical Data Manager (CDM) with over 25 years of experience
needs to build upon several foundational competencies while acquiring new and
refined skillsets to transition into a Clinical Data Scientist role. According
to the provided documents, these include:

- **Foundational Competencies**:
  - Attention to detail.
  - Therapeutic area knowledge.
  - Communication skills in articulating complex data findings to trial teams.
  - Systematic data review and trending.
  - Project Management.
  - Design of data collection tools.

- **New and Refined Skillset**:
  - Leadership and subject matter expertise in guiding staff transitions from
CDM to Clinical Data Science.
  - Influential and leadership skills, inc

## 💬 Block 8 -- Interactive Q&A Loop

Remembers the last `HISTORY_TURNS` Q&A pairs so follow-up questions work.
Set `HISTORY_TURNS = 0` in Block 7 to turn memory off.

Type your question and press Enter. Type `quit` to stop.


In [8]:
print("\U0001f9ed CDM Compass -- Interactive Q&A")
print(f"   Memory: last {HISTORY_TURNS} turns  |  type 'quit' to stop\n")

conversation_history = []

while True:
    question = input('\n\u2753 Your question: ').strip()
    if not question:
        print("   \u26a0\ufe0f  Please type a question.")
        continue
    if question.lower() in ("quit", "exit", "stop", "q"):
        print("\n\U0001f44b Goodbye!")
        break
    answer = ask_question(
        question, top_k=TOP_K, final_k=FINAL_K,
        history=conversation_history if HISTORY_TURNS > 0 else None,
    )
    conversation_history.append((question, answer))
    if len(conversation_history) > HISTORY_TURNS:
        conversation_history.pop(0)


🧭 CDM Compass -- Interactive Q&A
   Memory: last 3 turns  |  type 'quit' to stop


🧭 CDM Compass  --  "What can you tell me about data governance related to CDM or CDS?"
   70% semantic + 30% keyword  |  top 5 of 10
   5 chunks after reranking  (types: paragraph)
   ~489 tokens
   Asking qwen2.5:14b...

💬  ANSWER
According to ICH E6(R3) Step 4 Final Guideline [1], the responsible parties
(investigators and sponsors) are required to manage data integrity,
traceability, and security appropriately. This management ensures accurate
reporting, verification, and interpretation of clinical trial-related
information.

The FDA recommends including data related to prior therapies in clinical trial
data due to their significance [3]. This recommendation highlights the
importance of comprehensive documentation for thorough analysis and compliance
with regulatory standards.

------------------------------------------------------------
📚  SOURCES CITED  (in order cited -- gaps = retrieved but not us

---
## 📖 Quick Reference

### How CDM Compass finds answers
1. **Type-aware chunking** (Block 4) -- structured docs get hierarchical chunks; narrative gets paragraphs; tables get row-sentences
2. **Hybrid search** -- ChromaDB (meaning) + BM25 (keywords)
3. **Reranking** -- cross-encoder selects best `FINAL_K`
4. **Compression** -- most relevant sentences per chunk
5. **Context-aware answer** -- last `HISTORY_TURNS` Q&A pairs for follow-ups

### Chunk types shown in sources
| Type | Meaning |
|------|---------|
| `hier_section` | Full section from a structured document |
| `hier_leaf` | Individual paragraph within a section |
| `paragraph` | Paragraph from a narrative document |
| `table` | A table row converted to a readable sentence |

### Chunking settings (Block 4) -- re-run Blocks 4+5 after changing
| Setting | Default | Effect |
|---------|---------|--------|
| `MIN_PARA_WORDS` | 30 | Merge paragraphs shorter than this |
| `MAX_PARA_WORDS` | 400 | Split paragraphs longer than this |
| `HIER_SECTION_WORDS` | 800 | Max words in a section-level chunk |

### Search settings (Block 7)
| Setting | Default | Effect |
|---------|---------|--------|
| `TOP_K` | 10 | Candidates before reranking |
| `FINAL_K` | 5 | Chunks sent to LLM |
| `HISTORY_TURNS` | 3 | Conversation memory (0 = off) |
| `TOKEN_WARN_LIMIT` | 3000 | Warn if prompt exceeds this |
| `SEMANTIC_WEIGHT` | 0.7 | Raise for conceptual questions |
| `KEYWORD_WEIGHT` | 0.3 | Raise for exact terms, IDs |

### Relevance badges
| Badge | Hybrid score | Meaning |
|-------|-------------|---------|
| 🟢 HIGH | < 0.5 | Strong match |
| 🟡 MEDIUM | 0.5 - 0.65 | Partial match |
| 🔴 LOW | > 0.65 | Weak match |

### Managing document versions
| Situation | What to do |
|-----------|------------|
| Updated document | Delete old, drop in new, re-run Blocks 3-5 |
| Keep both versions | Rename with version numbers, re-run Blocks 3-5 |
